In [1]:
import pandas as pd
import geopandas as gpd
import os
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Point, Polygon, mapping
from fiona import collection
import matplotlib.pyplot as plt
import alphashape
import netCDF4 as nc
import xarray as xr
import regionmask
import scipy
from scipy import signal

In [2]:
def to_ts(file, dim, var, var_name, timeslice=None, detrend=False):
    '''
    extract a time series dataframe from netCDF file

    Parameters:
    - file (str): path to .nc data file
    - dim (list): input spatial dimension names (e.g., ['lat', 'lon'])
    - var (str): variable name in .nc
    - var_name (str): name for output time series variable

    Returns:
    - pd.DataFrame: DataFrame with time as index and column for the variable
    '''
    
    ds = xr.open_dataset(file)
    
    if timeslice is not None:
        ds = ds.sel(time=slice(timeslice[0], timeslice[1]))

    ds[var] = ds[var].astype(np.float64)
    ds_avg = ds.mean(dim=dim)
        
    if detrend:
        for j in range(12):
            ds_avg[var][j::12] = signal.detrend(ds_avg[var][j::12], axis=0)
    
    ts = ds_avg[var].to_pandas().rename(var_name)
    
    time = list(ds_avg['time'].values)
    var_values = list(ts.values)
    
    return pd.DataFrame({var_name: var_values}, index=time)

In [3]:
def to_ts2(ds, dim, var, var_name, timeslice=None, detrend=False):
    '''
    extract a time series dataframe from netCDF file

    Parameters:
    - ds (str): xarray object
    - dim (list): input spatial dimension names (e.g., ['lat', 'lon'])
    - var (str): variable name in .nc
    - var_name (str): name for output time series variable

    Returns:
    - pd.DataFrame: DataFrame with time as index and column for the variable
    '''
    
    if timeslice is not None:
        ds = ds.sel(time=slice(timeslice[0], timeslice[1]))

    ds[var] = ds[var].astype(np.float64)
    ds_avg = ds.mean(dim=dim)
    ds_avg = ds_avg.squeeze()
    
    if detrend:
        for j in range(12):
            ds_avg[var][j::12] = signal.detrend(ds_avg[var][j::12], axis=0)

    ts = ds_avg[var].to_pandas().rename(var_name)

    time = list(ds_avg['time'].values)
    var_values = list(ts.values)

    return pd.DataFrame({var_name: var_values}, index=time)

In [4]:
path = '/Users/ernest/Research/thesis/_paper_/data/'

### AMOC

In [5]:
sst_era5 = xr.open_dataset(path + 'AMOC/ERA5-monthly-global-SST.nc')
sst_hadcrut5 = xr.open_dataset(path + 'AMOC/HadCRUT5/HadCRUT.5.0.2.0.analysis.anomalies.ensemble_mean.nc')
sst_hadsst4 = xr.open_dataset(path + 'AMOC/HadSST4/HadSST.4.0.1.0_median.nc')
sst_cobe = xr.open_dataset(path + 'AMOC/COBE-SST2/sst.mon.mean.nc')
sst_ersst = xr.open_dataset(path + 'AMOC/noaa.ersst.v5/sst.mnmean.nc')

In [6]:
spg = gpd.read_file(path + 'shapefiles/spg_outline.shp')

In [7]:
spg_mask = regionmask.mask_3D_geopandas(spg, sst_era5.longitude, sst_era5.latitude)
spg_era5 = sst_era5.where(spg_mask)

spg_mask = regionmask.mask_3D_geopandas(spg, sst_hadcrut5.longitude, sst_hadcrut5.latitude)
spg_hadcrut5 = sst_hadcrut5.where(spg_mask)

spg_mask = regionmask.mask_3D_geopandas(spg, sst_hadsst4.longitude, sst_hadsst4.latitude)
spg_hadsst4 = sst_hadsst4.where(spg_mask)

spg_mask = regionmask.mask_3D_geopandas(spg, sst_cobe.lon, sst_cobe.lat)
spg_cobe = sst_cobe.where(spg_mask)

spg_mask = regionmask.mask_3D_geopandas(spg, sst_ersst.lon, sst_ersst.lat)
spg_ersst = sst_ersst.where(spg_mask)

In [8]:
ts_era5 = to_ts2(sst_era5, ['latitude','longitude'], 'sst', 'sst_glob',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
ts_spg_era5 = to_ts2(spg_era5, ['latitude','longitude'], 'sst', 'sst_spg',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)

ts_hadcrut5 = to_ts2(sst_hadcrut5, ['latitude','longitude'], 'tas_mean', 'sst_glob',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
ts_spg_hadcrut5 = to_ts2(spg_hadcrut5, ['latitude','longitude'], 'tas_mean', 'sst_spg',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)

ts_hadsst4 = to_ts2(sst_hadsst4, ['latitude','longitude'], 'tos', 'sst_glob',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
ts_spg_hadsst4 = to_ts2(spg_hadsst4, ['latitude','longitude'], 'tos', 'sst_spg',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)

ts_cobe = to_ts2(sst_cobe, ['lat','lon'], 'sst', 'sst_glob',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
ts_spg_cobe = to_ts2(spg_cobe, ['lat','lon'], 'sst', 'sst_spg',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)

ts_ersst = to_ts2(sst_ersst, ['lat','lon'], 'sst', 'sst_glob',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
ts_spg_ersst = to_ts2(spg_ersst, ['lat','lon'], 'sst', 'sst_spg',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)

In [9]:
AMOC_era5 = pd.concat([ts_spg_era5, ts_era5], axis=1)
AMOC_era5['AMOC_era5']= AMOC_era5['sst_spg'] - AMOC_era5['sst_glob']

AMOC_hadcrut5 = pd.concat([ts_spg_hadcrut5, ts_hadcrut5], axis=1)
AMOC_hadcrut5['AMOC_hadcrut5']= AMOC_hadcrut5['sst_spg'] - AMOC_hadcrut5['sst_glob']

AMOC_hadsst4 = pd.concat([ts_spg_hadsst4, ts_hadsst4], axis=1)
AMOC_hadsst4['AMOC_hadsst4']= AMOC_hadsst4['sst_spg'] - AMOC_hadsst4['sst_glob']

AMOC_cobe = pd.concat([ts_spg_cobe, ts_cobe], axis=1)
AMOC_cobe['AMOC_cobe']= AMOC_cobe['sst_spg'] - AMOC_cobe['sst_glob']

AMOC_ersst = pd.concat([ts_spg_ersst, ts_ersst], axis=1)
AMOC_ersst['AMOC_ersst']= AMOC_ersst['sst_spg'] - AMOC_ersst['sst_glob']

In [10]:
AMOC_hadcrut5 = AMOC_hadcrut5.resample('M').mean()
AMOC_hadcrut5.index = AMOC_hadcrut5.index + pd.offsets.MonthBegin(-1)

AMOC_hadsst4 = AMOC_hadsst4.resample('M').mean()
AMOC_hadsst4.index = AMOC_hadsst4.index + pd.offsets.MonthBegin(-1)

### CLLJ

In [11]:
# units: m/s
CLLJ = to_ts(path + 'CLLJ/cdo-fldmean/CLLJ_u-wind_aggregated.nc', ['lat','lon'], 'u', 'CLLJ', 
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
CLLJ = -CLLJ

### AR precipitation

In [12]:
# ERA5 (m of water equivalent per day)
# tp [mm] = tp [m/day]⋅1000⋅N
ERA5_prec = to_ts(path + 'AR/prec/ERA5_south.nc', ['latitude','longitude'], 'tp', 'SAR_prec', 
                      timeslice=('1982-01-01', '2022-12-31'), detrend=True)

### AR NDVI

In [13]:
NDVI_bw = to_ts(path + 'AR/ndvi/south.nc4',
      ['lat','lon'], 'ndvi', 'NDVI', timeslice=('1982-01-01', '2022-12-31'), detrend=True)

# resample + reindex
NDVI = NDVI_bw.resample('M').mean()
NDVI.index = NDVI.index + pd.offsets.MonthBegin(-1)

### mask

In [14]:
# SAR dry season (MJJAS)
pattern = [1,1,1,1,0,0,0,0,0,1,1,1] * 41

## other indicators

### ENSO

In [15]:
ENSO12_data = xr.open_dataset(path + 'other/ENSO1+2.nc')
ENSO34_data = xr.open_dataset(path + 'other/ENSO34.nc')

In [16]:
ENSO12 = to_ts2(ENSO12_data, ['lat', 'lon'], 'sst', 'ENSO1+2', timeslice=('1982-01-01', '2022-12-31'), detrend=False)
ENSO34 = to_ts2(ENSO34_data, ['lat', 'lon'], 'sst', 'ENSO34', timeslice=('1982-01-01', '2022-12-31'), detrend=False)

### ITCZ

In [17]:
itcz_data = xr.open_dataset(path + 'other/ITCZ-box.nc')
itcz_data = itcz_data.sel(time=slice('1982-01-01', '2022-12-31'))

In [18]:
precip_weighted_lat_means = []

for i in range(len(itcz_data.time)):
    # Select data for the current time step
    region = itcz_data.isel(time=i)

    # Calculate zonal mean precipitation
    zm = region['tp'].mean(dim='longitude')

    # Multiply zonal mean precipitation by latitude values
    weighted_lat = zm * region['latitude']

    # Calculate precipitation-weighted latitude mean for the current time step
    precip_weighted_lat_mean = (weighted_lat.sum(dim='latitude') / zm.sum(dim='latitude')).values

    # Append the result to the list
    precip_weighted_lat_means.append(precip_weighted_lat_mean)

# Convert the list to a numpy array
ITCZ_index = np.array(precip_weighted_lat_means)

In [19]:
time_cycle = 12

# detrend
for j in range(time_cycle):
    ITCZ_index[j::time_cycle] = signal.detrend(ITCZ_index[j::time_cycle], axis = 0) 

### NATL + SATL

In [20]:
NATL_data = xr.open_dataset(path + 'other/NATL-sst.nc')
SATL_data = xr.open_dataset(path + 'other/SATL-sst.nc')

In [21]:
NATL = to_ts2(NATL_data, ['latitude','longitude'], 'sst', 'NATL',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)
SATL = to_ts2(SATL_data, ['latitude','longitude'], 'sst', 'SATL',
             timeslice=('1982-01-01', '2022-12-31'), detrend=True)

# calculate ANSG before detrending NATL and SATL
working_NATL = to_ts2(NATL_data, ['latitude','longitude'], 'sst', 'ANSG',
             timeslice=('1982-01-01', '2022-12-31'), detrend=False)
working_SATL = to_ts2(SATL_data, ['latitude','longitude'], 'sst', 'ANSG',
             timeslice=('1982-01-01', '2022-12-31'), detrend=False)
ANSG = working_NATL - working_SATL

# detrend ANSG
for j in range(12):
            ANSG[j::12] = signal.detrend(ANSG[j::12], axis=0)

### SAA

In [22]:
SAA = xr.open_dataset(path + 'other/SAA-box.nc')
SAA = SAA.sel(time=slice('1982-01-01', '2022-12-31'))

In [23]:
slp_data = SAA.variables['msl'][:]
latitudes = SAA.variables['latitude'][:]
longitudes = SAA.variables['longitude'][:]

# Initialize lists to store selected data for each time step
selected_latitudes_list = []
selected_longitudes_list = []
selected_slp_values_list = []

# Loop over each time step
for timestep in range(slp_data.shape[0]):
    # Calculate the mean SLP within the preliminary boundary for the current time step
    mean_slp = np.mean(slp_data[timestep])
    
    # Select grid points where SLP is greater than the monthly average for the current time step
    selected_slp_indices = np.where(slp_data[timestep] > mean_slp)
    
    # Extract the latitude, longitude, and SLP values for the current time step
    selected_latitudes = latitudes[selected_slp_indices[0]]
    selected_longitudes = longitudes[selected_slp_indices[1]]
    selected_slp_values = slp_data[timestep, selected_slp_indices[0], selected_slp_indices[1]]
    
    # Append selected data for the current time step to the lists
    selected_latitudes_list.append(selected_latitudes)
    selected_longitudes_list.append(selected_longitudes)
    selected_slp_values_list.append(selected_slp_values)

In [24]:
center_latitudes = []
center_longitudes = []

# Iterate over each time step
for i in range(len(SAA.time)):

    current = SAA.isel(time=i)
    
    # Calculate the mean SLP within the preliminary boundary
    mean_slp = np.mean(current.msl)

    # Select grid points where SLP is greater than the monthly average
    selected_msl = current.msl.where(current.msl > mean_slp, drop=True)
    
    mean_C = selected_msl.mean()
    C_msl = selected_msl.where(selected_msl > mean_C, drop=True)
    mean_D = C_msl.mean()
    D_msl = C_msl.where(C_msl > mean_D, drop=True)

    mean_E = D_msl.mean()
    std_E = D_msl.std()
    E_msl = D_msl.where(D_msl > mean_E + std_E, drop=True)

    # Extract latitude and longitude coordinates from E_msl
    center_latitude = E_msl.latitude.mean().values
    center_longitude = E_msl.longitude.mean().values

    # Store mean latitude and longitude in center_latitudes and center_longitudes
    center_latitudes.append(center_latitude)
    center_longitudes.append(center_longitude)

In [25]:
for j in range(12):
            center_latitudes[j::12] = signal.detrend(center_latitudes[j::12], axis=0)
            center_longitudes[j::12] = signal.detrend(center_longitudes[j::12], axis=0)

### NAO

In [26]:
NAO_data = pd.read_csv(path + 'other/norm.nao.monthly.b5001.current.ascii', delim_whitespace=True, header=None)
NAO=list(NAO_data[2][384:876])

## merge + re-index

### ERA5 (1982-2022)

In [27]:
allvars = pd.concat([AMOC_era5['AMOC_era5'], AMOC_hadcrut5['AMOC_hadcrut5'], 
                  AMOC_hadsst4['AMOC_hadsst4'], AMOC_cobe['AMOC_cobe'], 
                  AMOC_ersst['AMOC_ersst'], CLLJ, ERA5_prec, NDVI, 
                  ENSO12, ENSO34, NATL, SATL, ANSG], axis=1)

allvars = allvars.reset_index().rename(columns={'index': 'time'})
allvars['ITCZ'] = ITCZ_index
allvars['SAA_lat'] = center_latitudes
allvars['SAA_lon'] = center_longitudes
allvars['NAO'] = NAO
allvars['Southern AR dry season'] = pattern
allvars

,time,AMOC_era5,AMOC_hadcrut5,AMOC_hadsst4,AMOC_cobe,AMOC_ersst,CLLJ,SAR_prec,NDVI,ENSO1+2,ENSO34,NATL,SATL,ANSG,ITCZ,SAA_lat,SAA_lon,NAO,Southern AR dry season
0,1982-01-01,0.573397,1.084507,0.479555,0.536978,0.611327,-0.365887,0.000504,-168.179598,0.179779,0.283997,0.362343,-0.444039,0.806382,0.436565,0.512196,-2.968779,-0.8900,1
1,1982-02-01,0.336767,0.056201,0.195116,0.295782,0.280656,0.039025,0.000249,-231.937396,0.125458,0.247711,0.224525,-0.815832,1.040357,0.185830,1.554443,-10.731843,1.1500,1
2,1982-03-01,0.074182,0.341188,0.291707,0.196003,0.228871,0.657247,-0.000834,-115.396364,-0.399200,0.793549,0.042072,-0.696431,0.738504,0.456012,-1.402477,-2.288367,1.1500,1
3,1982-04-01,-0.029423,-0.014307,-0.068145,0.121964,-0.005972,0.104014,0.000448,-93.576626,-0.531616,0.948547,-0.110762,-0.368460,0.257698,0.784051,-2.363632,6.303261,0.1000,1
4,1982-05-01,-0.160463,-0.167537,-0.274569,-0.039897,0.013368,-1.868915,-0.000363,-199.433349,0.202026,1.005524,-0.106411,-0.383254,0.276843,0.439906,5.358128,-8.324227,-0.5300,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,2022-08-01,0.155575,-0.045180,0.000726,0.265877,0.072585,0.173786,0.000286,-136.678083,-0.233582,-0.365387,-0.120345,-0.025048,-0.095297,0.065300,-2.765247,-10.521786,1.4699,0
488,2022-09-01,0.876525,0.870340,0.672905,0.814319,0.597346,-0.997812,-0.000310,-335.963465,-0.836182,-0.332672,-0.092767,-0.063172,-0.029595,-0.014989,0.219942,1.483561,-1.6105,0
489,2022-10-01,0.660696,0.484523,0.503241,0.506393,0.422334,0.377371,0.000110,-185.535830,-1.382172,-0.426758,-0.171207,0.117667,-0.288874,0.153792,1.313581,-4.896777,-0.7175,1
490,2022-11-01,-0.075736,0.536517,-0.209032,-0.174987,-0.264646,-0.219247,-0.002352,244.348158,-1.003448,-0.381104,-0.193416,0.065157,-0.258573,-0.758287,-3.816204,-18.536150,0.6922,1


In [28]:
allvars.to_csv(path + 'ERA5_robustness-allvars.csv')